# LLM Connectivity Benchmark

This notebook benchmarks `gemma3:12b-it-qat` against the Week 3 GNN connectivity output. It derives top connected undirected node pairs from the saved Week 3 edge tensor and compares them to strict-JSON LLM predictions on the same 12-hour windows.

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI

TEAM_ROOT = Path.cwd().resolve()
if TEAM_ROOT.name == 'Notebooks':
    TEAM_ROOT = TEAM_ROOT.parent
elif not (TEAM_ROOT / 'Prompts').exists():
    TEAM_ROOT = Path('/Users/bara407/projects/GenAICompetition/ESSD_AI_Competition_1/SubSignal_Week_4_LLM_Gemma/team_folder')

REPO_ROOT = TEAM_ROOT.parents[2]
WEEK3_DATA_CSV = REPO_ROOT / 'ESSD_AI_Competition_1' / 'SubSignal_Week_3' / 'Data' / 'ftes_scaled_for_GNN.csv'
WEEK3_EDGE_NPY = REPO_ROOT / 'ESSD_AI_Competition_1' / 'SubSignal_Week_3' / 'Scripts' / 'edge_weights_over_time.npy'
PROMPT_TEMPLATE_PATH = TEAM_ROOT / 'Prompts' / 'prompt_template.txt'
FEW_SHOT_PATH = TEAM_ROOT / 'Prompts' / 'few_shot_samples.json'
RESULTS_DIR = TEAM_ROOT / 'Results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'gemma3:12b-it-qat'
MODEL_LABEL = 'gemma3_12b_it_qat'
ITERATION_NUMBER = 1
TOP_K = 3
WINDOW_SIZE = 12
PURGE_GAP_WINDOWS = 12
OLLAMA_BASE_URL = os.environ.get('OLLAMA_OPENAI_BASE_URL', 'http://localhost:11434/v1/')
OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY', 'ollama')
SYSTEM_PROMPT = 'You are a careful connectivity analyst that follows JSON output instructions exactly.'

NODE_COLUMNS = [
    'tl_interval_pressure', 'tl_bottom_pressure',
    'tn_interval_pressure', 'tn_bottom_pressure',
    'tc_interval_pressure', 'tc_bottom_pressure',
    'tu_interval_pressure', 'tu_bottom_pressure',
    'ts_interval_pressure', 'ts_bottom_pressure',
]

RAW_RESULTS_PATH = RESULTS_DIR / f'{MODEL_LABEL}_results_raw_{ITERATION_NUMBER}.csv'
CLEAN_RESULTS_PATH = RESULTS_DIR / f'{MODEL_LABEL}_results_clean_{ITERATION_NUMBER}.csv'

print(TEAM_ROOT)
print(WEEK3_DATA_CSV)
print(WEEK3_EDGE_NPY)
print(OLLAMA_BASE_URL)

In [ ]:
def build_pair_id(name_a: str, name_b: str) -> str:
    left, right = sorted((name_a, name_b))
    return f'{left}__{right}'


def build_undirected_pair_ids(node_columns: list[str]) -> list[str]:
    pair_ids = []
    for i in range(len(node_columns)):
        for j in range(i + 1, len(node_columns)):
            pair_ids.append(build_pair_id(node_columns[i], node_columns[j]))
    return pair_ids


def summarize_series(name: str, series: pd.Series) -> str:
    start = float(series.iloc[0])
    end = float(series.iloc[-1])
    delta = end - start
    mean = float(series.mean())
    std = float(series.std(ddof=0))
    return f'{name}: start={start:.3f}, end={end:.3f}, delta={delta:+.3f}, mean={mean:.3f}, std={std:.3f}'


def derive_regime_label(window_df: pd.DataFrame) -> str:
    injecting = int(window_df['tc_injecting'].max())
    producing = int(window_df['tc_producing'].max())
    if injecting and producing:
        return 'mixed'
    if producing:
        return 'producing'
    if injecting:
        return 'injecting'
    return 'idle'


def build_window_summary(window_df: pd.DataFrame) -> str:
    lines = [
        summarize_series('net_flow', window_df['net_flow']),
        summarize_series('injection_pressure', window_df['injection_pressure']),
        summarize_series('delta_P_delta_Q', window_df['delta_P_delta_Q']),
        f"tc_injecting_max: {int(window_df['tc_injecting'].max())}",
        f"tc_producing_max: {int(window_df['tc_producing'].max())}",
        'node_pressure_traces:',
    ]
    for column in NODE_COLUMNS:
        lines.append(summarize_series(column, window_df[column]))
    return '\n'.join(lines)


def derive_window_benchmark_df(df: pd.DataFrame, edge_tensor: np.ndarray) -> pd.DataFrame:
    last_epoch = edge_tensor[-1]
    directed_edges = [(i, j) for i in range(len(NODE_COLUMNS)) for j in range(len(NODE_COLUMNS)) if i != j]
    edge_index = {edge: idx for idx, edge in enumerate(directed_edges)}
    records: list[dict[str, object]] = []
    for window_id in range(last_epoch.shape[0]):
        window_df = df.iloc[window_id:window_id + WINDOW_SIZE].copy()
        scores: list[tuple[str, float]] = []
        for i in range(len(NODE_COLUMNS)):
            for j in range(i + 1, len(NODE_COLUMNS)):
                score = (
                    abs(last_epoch[window_id, edge_index[(i, j)]])
                    + abs(last_epoch[window_id, edge_index[(j, i)]])
                ) / 2.0
                scores.append((build_pair_id(NODE_COLUMNS[i], NODE_COLUMNS[j]), float(score)))
        scores.sort(key=lambda item: (-item[1], item[0]))
        records.append(
            {
                'window_id': window_id,
                'start_time': str(window_df.iloc[0]['time']),
                'end_time': str(window_df.iloc[-1]['time']),
                'regime_label': derive_regime_label(window_df),
                'window_summary_text': build_window_summary(window_df),
                'gnn_top_pairs': [pair for pair, _score in scores[:TOP_K]],
                'gnn_pair_scores': {pair: score for pair, score in scores[:10]},
            }
        )
    return pd.DataFrame(records)


def add_benchmark_splits(df: pd.DataFrame, purge_gap: int = PURGE_GAP_WINDOWS) -> pd.DataFrame:
    out = df.copy()
    out['split'] = 'unused'
    n_rows = len(out)
    prompt_dev_end = int(n_rows * 0.60)
    validation_end = int(n_rows * 0.80)

    out.loc[: prompt_dev_end - 1, 'split'] = 'prompt_dev'
    out.loc[prompt_dev_end : prompt_dev_end + purge_gap - 1, 'split'] = 'purged_after_prompt_dev'

    val_start = prompt_dev_end + purge_gap
    out.loc[val_start : validation_end - 1, 'split'] = 'validation'
    out.loc[validation_end : validation_end + purge_gap - 1, 'split'] = 'purged_after_validation'

    test_start = validation_end + purge_gap
    out.loc[test_start:, 'split'] = 'test'
    return out


def format_few_shot_block(samples: list[dict[str, object]]) -> str:
    blocks = []
    for sample in samples:
        blocks.append(
            '\n'.join(
                [
                    f"Example {sample['example_id']}:",
                    f"window_id: {sample['window_id']}",
                    f"regime_label: {sample['regime_label']}",
                    'Window summary:',
                    sample['window_summary_text'],
                    f"Expected JSON: {json.dumps(sample['output_json'])}",
                ]
            )
        )
    return '\n\n'.join(blocks)


def build_prompt(row: pd.Series, prompt_template: str, allowed_pair_ids: list[str], few_shot_samples: list[dict[str, object]]) -> str:
    return prompt_template.format(
        top_k=TOP_K,
        allowed_pair_ids_json=json.dumps(allowed_pair_ids),
        few_shot_block=format_few_shot_block(few_shot_samples),
        window_id=row['window_id'],
        start_time=row['start_time'],
        end_time=row['end_time'],
        regime_label=row['regime_label'],
        window_summary_text=row['window_summary_text'],
    )


def parse_response(raw_text: str, allowed_pair_ids: set[str]) -> dict[str, object]:
    result = {
        'parsed_top_pairs': None,
        'confidence': np.nan,
        'rationale': None,
        'parse_ok': False,
        'parse_error': None,
    }
    if not isinstance(raw_text, str) or not raw_text.strip():
        result['parse_error'] = 'empty_response'
        return result

    match = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = match.group(0) if match else raw_text.strip()
    try:
        payload = json.loads(candidate)
    except Exception:
        result['parse_error'] = 'invalid_json'
        return result

    top_pairs = payload.get('top_pairs')
    if not isinstance(top_pairs, list):
        result['parse_error'] = 'missing_top_pairs'
        return result

    normalized_pairs = [str(pair).strip() for pair in top_pairs]
    if len(normalized_pairs) != TOP_K:
        result['parse_error'] = 'wrong_pair_count'
        return result
    if len(set(normalized_pairs)) != TOP_K:
        result['parse_error'] = 'duplicate_pairs'
        return result
    if any(pair not in allowed_pair_ids for pair in normalized_pairs):
        result['parse_error'] = 'invalid_pair_id'
        return result

    try:
        confidence = float(payload.get('confidence'))
    except Exception:
        result['parse_error'] = 'invalid_confidence'
        return result
    if not 0.0 <= confidence <= 1.0:
        result['parse_error'] = 'invalid_confidence'
        return result

    result['parsed_top_pairs'] = normalized_pairs
    result['confidence'] = confidence
    result['rationale'] = str(payload.get('rationale', '')).strip()
    result['parse_ok'] = True
    return result


def score_prediction(target_pairs: list[str], predicted_pairs: list[str] | None) -> dict[str, float]:
    if not predicted_pairs:
        return {'top3_overlap_rate': 0.0, 'jaccard_at_3': 0.0, 'top1_match_rate': 0.0}
    target_set = set(target_pairs)
    pred_set = set(predicted_pairs)
    intersection = len(target_set & pred_set)
    union = len(target_set | pred_set)
    return {
        'top3_overlap_rate': intersection / len(target_pairs),
        'jaccard_at_3': intersection / union if union else 0.0,
        'top1_match_rate': float(predicted_pairs[0] == target_pairs[0]),
    }


def get_client() -> OpenAI:
    return OpenAI(api_key=OLLAMA_API_KEY, base_url=OLLAMA_BASE_URL)


def query_llm(prompt_text: str, temperature: float = 0.0, seed: int = 0) -> str:
    client = get_client()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt_text},
        ],
        temperature=temperature,
        seed=seed,
    )
    return response.choices[0].message.content or ''


ALLOWED_PAIR_IDS = build_undirected_pair_ids(NODE_COLUMNS)
ALLOWED_PAIR_ID_SET = set(ALLOWED_PAIR_IDS)
len(ALLOWED_PAIR_IDS)

In [ ]:
week3_df = pd.read_csv(WEEK3_DATA_CSV)
edge_tensor = np.load(WEEK3_EDGE_NPY)
benchmark_df = derive_window_benchmark_df(week3_df, edge_tensor)
benchmark_df = add_benchmark_splits(benchmark_df)

print(edge_tensor.shape)
print(benchmark_df['split'].value_counts().to_string())
display(benchmark_df[['window_id', 'start_time', 'end_time', 'regime_label', 'split', 'gnn_top_pairs']].head())

In [ ]:
prompt_template = PROMPT_TEMPLATE_PATH.read_text()
few_shot_samples = json.loads(FEW_SHOT_PATH.read_text())

parser_smoke_cases = [
    (json.dumps({'top_pairs': ALLOWED_PAIR_IDS[:3], 'confidence': 0.7, 'rationale': 'ok'}), True),
    ('not json', False),
    (json.dumps({'top_pairs': ['bad_pair', ALLOWED_PAIR_IDS[1], ALLOWED_PAIR_IDS[2]], 'confidence': 0.7, 'rationale': 'bad'}), False),
    (json.dumps({'top_pairs': ALLOWED_PAIR_IDS[:2], 'confidence': 0.7, 'rationale': 'short'}), False),
]

for raw_text, expected_ok in parser_smoke_cases:
    parsed = parse_response(raw_text, ALLOWED_PAIR_ID_SET)
    print(expected_ok, parsed['parse_ok'], parsed['parse_error'])

sample_prompt = build_prompt(benchmark_df.iloc[0], prompt_template, ALLOWED_PAIR_IDS, few_shot_samples)
print(sample_prompt[:3000])

In [ ]:
smoke_row = benchmark_df.loc[benchmark_df['split'] == 'validation'].iloc[0].copy()
smoke_prompt = build_prompt(smoke_row, prompt_template, ALLOWED_PAIR_IDS, few_shot_samples)

try:
    smoke_raw = query_llm(smoke_prompt)
    smoke_parsed = parse_response(smoke_raw, ALLOWED_PAIR_ID_SET)
    print('Raw response:')
    print(smoke_raw)
    print('Parsed response:')
    print(smoke_parsed)
except Exception as exc:
    print(f'Smoke test could not reach the local Ollama endpoint: {exc}')

In [ ]:
test_df = benchmark_df.loc[benchmark_df['split'] == 'test'].copy().reset_index(drop=True)
rows = []

for idx, row in test_df.iterrows():
    prompt_text = build_prompt(row, prompt_template, ALLOWED_PAIR_IDS, few_shot_samples)
    raw_response = ''
    parsed = {
        'parsed_top_pairs': None,
        'confidence': np.nan,
        'rationale': None,
        'parse_ok': False,
        'parse_error': None,
    }
    try:
        raw_response = query_llm(prompt_text)
        parsed = parse_response(raw_response, ALLOWED_PAIR_ID_SET)
    except Exception as exc:
        parsed['parse_error'] = f'request_failed: {exc}'

    scores = score_prediction(row['gnn_top_pairs'], parsed['parsed_top_pairs'])
    rows.append(
        {
            'window_id': int(row['window_id']),
            'start_time': row['start_time'],
            'end_time': row['end_time'],
            'split': row['split'],
            'regime_label': row['regime_label'],
            'prompt_text': prompt_text,
            'gnn_top_pairs': row['gnn_top_pairs'],
            'raw_response': raw_response,
            **parsed,
            **scores,
        }
    )

    if (idx + 1) % 25 == 0:
        print(f'Completed {idx + 1}/{len(test_df)} windows')

results_df = pd.DataFrame(rows)
results_df['gnn_top_pairs'] = results_df['gnn_top_pairs'].apply(json.dumps)
results_df['parsed_top_pairs'] = results_df['parsed_top_pairs'].apply(lambda value: json.dumps(value) if isinstance(value, list) else '')
results_df.to_csv(RAW_RESULTS_PATH, index=False)

clean_cols = [
    'window_id', 'start_time', 'end_time', 'split', 'regime_label',
    'gnn_top_pairs', 'parsed_top_pairs', 'confidence', 'parse_ok', 'parse_error',
    'top3_overlap_rate', 'jaccard_at_3', 'top1_match_rate',
]
results_df[clean_cols].to_csv(CLEAN_RESULTS_PATH, index=False)

print(RAW_RESULTS_PATH)
print(CLEAN_RESULTS_PATH)
display(results_df.head())

In [ ]:
if CLEAN_RESULTS_PATH.exists():
    clean_df = pd.read_csv(CLEAN_RESULTS_PATH)
    summary = clean_df.groupby(['split', 'regime_label'], as_index=False).agg(
        rows=('window_id', 'size'),
        parse_success_rate=('parse_ok', 'mean'),
        top3_overlap_rate=('top3_overlap_rate', 'mean'),
        jaccard_at_3=('jaccard_at_3', 'mean'),
        top1_match_rate=('top1_match_rate', 'mean'),
    ).sort_values(['split', 'regime_label'])
    display(summary)
    overall = clean_df[['parse_ok', 'top3_overlap_rate', 'jaccard_at_3', 'top1_match_rate']].mean(numeric_only=True)
    print(overall.to_string())
else:
    print('Run the benchmark cell first.')